# Diffusion Model - Single Sample Testing
## Image Generation Using Diffusion Models

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from rollNumber_05 import *

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model configuration
IMAGE_SIZE = 64
T = 1000

# Initialize model
model = UNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    time_emb_dim=256,
    num_res_blocks=2
).to(device)

# Initialize forward process
forward_process = DiffusionForwardProcess(T=T, device=device)

# Create diffusion model
diffusion = DiffusionModel(model, forward_process, device)

# Load trained weights
checkpoint_path = 'models/best_model.pth'
checkpoint = torch.load(checkpoint_path, map_location=device)
diffusion.model.load_state_dict(checkpoint['model_state_dict'])
diffusion.model.eval()

print(f"Model loaded successfully from epoch {checkpoint['epoch']+1}")
print(f"Training loss: {checkpoint['loss']:.4f}")

In [2]:
# Generate samples
def generate_sample():
    with torch.no_grad():
        # Generate 1 sample from noise
        samples, _ = diffusion.sample(1, show_progress=True)
        
        # Convert to displayable format
        img = samples[0].cpu()
        img = (img + 1) / 2  # Denormalize from [-1, 1] to [0, 1]
        img = torch.clamp(img, 0, 1)
        img = img.permute(1, 2, 0).numpy()
        
        return img

# Generate and display multiple samples
num_samples = 4
fig, axes = plt.subplots(1, num_samples, figsize=(16, 4))

for i in range(num_samples):
    img = generate_sample()
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Generated Image {i+1}')

plt.tight_layout()
plt.savefig('generated_samples.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

In [3]:
# Display forward process on a sample image
def display_forward_process():
    # Create a synthetic image or load one
    # For demonstration, we'll create a random image
    sample_image = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    
    steps = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
    
    fig, axes = plt.subplots(1, len(steps), figsize=(20, 3))
    
    with torch.no_grad():
        for i, t in enumerate(steps):
            x_t, _ = diffusion.forward_process.add_noise(sample_image, torch.tensor([t], device=device))
            img = x_t[0].cpu()
            img = (img + 1) / 2
            img = torch.clamp(img, 0, 1)
            img = img.permute(1, 2, 0).numpy()
            
            axes[i].imshow(img)
            axes[i].set_title(f't={t}')
            axes[i].axis('off')
    
    plt.suptitle('Forward Diffusion Process', fontsize=14)
    plt.tight_layout()
    plt.savefig('forward_process_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

display_forward_process()

NameError: name 'torch' is not defined

In [ ]:
# Display reverse process for a single sample
def display_reverse_process():
    # Start from noise
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    
    # Store intermediate steps
    intermediate_steps = []
    steps_to_show = [999, 800, 600, 400, 200, 100, 0]
    
    with torch.no_grad():
        for t in range(999, -1, -1):
            t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
            noise_pred = diffusion.predict_noise(x, t_tensor)
            x = diffusion.denoise_step(x, t_tensor, noise_pred)
            
            if t in steps_to_show:
                intermediate_steps.append((t, x.clone()))
    
    # Display the reverse process
    fig, axes = plt.subplots(1, len(intermediate_steps), figsize=(20, 3))
    
    for i, (t, img_tensor) in enumerate(intermediate_steps):
        img = img_tensor[0].cpu()
        img = (img + 1) / 2
        img = torch.clamp(img, 0, 1)
        img = img.permute(1, 2, 0).numpy()
        
        axes[i].imshow(img)
        axes[i].set_title(f't={t}')
        axes[i].axis('off')
    
    plt.suptitle('Reverse Diffusion Process (Denoising)', fontsize=14)
    plt.tight_layout()
    plt.savefig('reverse_process_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

display_reverse_process()